
# HODGE v10a.29 — Blind \(m_5/m_6/m_7\) A100 Runner

This notebook is a **runner / hardening layer** for the existing project code:

- `NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb`
- `ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py`

It does **not** contain any published \(m_5,m_6,m_7\) target values.

Default workflow:

1. load/execute the v10a.26 base notebook into this kernel;
2. run the v10a.28 short firewall at order 5;
3. if every firewall gate passes, run blind order-5 production on the A100;
4. checkpoint every completed rooted shape;
5. freeze the blind \(m_5\) result and provenance;
6. block \(m_6\) until you explicitly mark \(m_5\) externally validated;
7. block \(m_7\) until you explicitly mark \(m_6\) externally validated.

The v10a.28 engine uses order-aware Krylov depth, dynamic SU(3) delta/epsilon Haar projectors, cached contraction paths, rooted support census, and canonical SW/BCH. This notebook does not change those physics routines; it controls execution, provenance, checkpointing, and the blind holdout sequence.

**Important:** production coefficients remain floating-point computer-assisted results until separately adjudicated by an exact rational backend.


In [ ]:

from __future__ import annotations

import os, sys, json, time, hashlib, platform, subprocess, shutil, zipfile
from pathlib import Path
from datetime import datetime, timezone

# ---------- USER SETTINGS ----------
USE_GOOGLE_DRIVE = True
WORKDIR_NAME = "HODGE_BLIND_M5_M7"
AUTO_UPLOAD_MISSING_FILES = True

V26_NAME = "NB_O4_hodge_v10a26_factor52complete_exactsw_rootedoracle_a100.ipynb"
V28_NAME = "ENGINE_O4_hodge_v10a28_orderaware_gram_firewall_a100.py"

# m5 production defaults for an A100.
M5_MAX_NEW_SHAPES = 0        # 0 = unlimited in one invocation
M5_TIME_BUDGET_MINUTES = 0   # 0 = no between-shape budget
GPU_SW_MIN_DIM = 64
HERM_AUDIT_PAIRS = 24
DUPLICATE_CHECKS = 1
HEARTBEAT_SECONDS = 20

print("UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())


In [ ]:

# A100 / CUDA environment check. Installs only missing lightweight dependencies.
def ensure_import(import_name, pip_name=None):
    try:
        return __import__(import_name)
    except Exception:
        if pip_name is None:
            raise
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return __import__(import_name)

np = ensure_import("numpy", "numpy")
sp = ensure_import("sympy", "sympy")
oe = ensure_import("opt_einsum", "opt_einsum")

try:
    import cupy as cp
except Exception:
    print("CuPy not importable; installing cupy-cuda12x.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"])
    import cupy as cp

gpu_count = int(cp.cuda.runtime.getDeviceCount())
if gpu_count < 1:
    raise RuntimeError("No CUDA GPU detected. Select an A100 GPU runtime before continuing.")

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"].decode() if isinstance(props["name"], (bytes, bytearray)) else str(props["name"])
free_b, total_b = cp.cuda.runtime.memGetInfo()

print("CUDA devices:", gpu_count)
print("GPU 0:", name)
print(f"GPU memory: free={free_b/2**30:.2f} GiB / total={total_b/2**30:.2f} GiB")
if "A100" not in name.upper():
    print("WARNING: runtime is not reporting an A100; the code can still run, but this notebook was tuned for A100.")


In [ ]:

# Mount Drive for durable checkpoints, or fall back to /content.
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        WORKDIR = Path("/content/drive/MyDrive") / WORKDIR_NAME
    except Exception as exc:
        print("Drive mount unavailable; using /content:", exc)
        WORKDIR = Path("/content") / WORKDIR_NAME
else:
    WORKDIR = Path("/content") / WORKDIR_NAME

WORKDIR.mkdir(parents=True, exist_ok=True)
print("WORKDIR:", WORKDIR)


In [ ]:

# Locate or upload the two source artifacts.
CONTENT = Path("/content")

def locate(name: str) -> Path | None:
    candidates = [
        CONTENT / name,
        WORKDIR / name,
        Path.cwd() / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def upload_if_missing(name: str) -> Path:
    p = locate(name)
    if p is not None:
        return p
    if not AUTO_UPLOAD_MISSING_FILES:
        raise FileNotFoundError(name)
    try:
        from google.colab import files
    except Exception as exc:
        raise FileNotFoundError(f"{name} not found and Colab upload is unavailable") from exc
    print(f"Upload: {name}")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(f"No file uploaded for {name}")
    # Accept exact file name first; otherwise accept a single uploaded file and rename it.
    if name in uploaded:
        p = CONTENT / name
    elif len(uploaded) == 1:
        src_name = next(iter(uploaded))
        src = CONTENT / src_name
        p = CONTENT / name
        if src != p:
            shutil.move(str(src), str(p))
    else:
        raise RuntimeError(f"Expected {name}; received {list(uploaded)}")
    return p

V26_PATH = upload_if_missing(V26_NAME)
V28_PATH = upload_if_missing(V28_NAME)

print("v10a.26:", V26_PATH)
print("v10a.28:", V28_PATH)


In [ ]:

# Provenance hashes. No requested-order target values are loaded.
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

V26_SHA256 = sha256_file(V26_PATH)
V28_SHA256 = sha256_file(V28_PATH)

print("v10a.26 SHA-256:", V26_SHA256)
print("v10a.28 SHA-256:", V28_SHA256)

src28 = V28_PATH.read_text(encoding="utf-8", errors="strict")
assert "target coefficient           : NOT LOADED" in src28
assert "external target remains absent" in src28
print("Blind-source declaration found in v10a.28.")



## Load the v10a.26 base namespace

The frontier source requires the completed v10a.26 namespace. The cell below executes the code cells of the uploaded v10a.26 notebook **inside this kernel**.

If the required symbols already exist, it does nothing.

This can be long if your v10a.26 notebook itself launches production. Watch its output and stop if the source file is not the completed/certified version you intend to use.


In [ ]:

import nbformat

V28_REQUIRED_GLOBALS = (
    "L", "N", "faces", "verts", "T1_POLS", "anchor_faces", "V23C_ROOT",
    "V23C_POL", "_FAST_EPS", "oe", "LXState", "_V17_VAC",
    "_v17_apply_W_faces", "_v17_apply_W_labeled", "_v17_connected",
    "_v17_phys_index", "_v17_translate_support", "_v17_translate_face",
    "_v23c_split_h0", "_v23c_rooted_connected_subsets", "_v24c_shape_key",
    "_v24c_candidate_supports", "_v10a3_face_state", "_v10a3_h0_state_inner",
    "_v10a3_physical_blocks", "_v10a3_compress_state", "_v9_flux_key_state",
    "_joint_canon_states", "lx_combine_bra_ket", "_v26_sw_blocks",
    "_v23_sw_exact", "_v23_sp", "_v23_random", "_V23CF",
    "_v26_singlet_multiplicity", "_V17_NEIGH", "_v23c_fit_cluster",
)

def missing_base_symbols():
    return [x for x in V28_REQUIRED_GLOBALS if x not in globals()]

def execute_notebook_in_current_kernel(path: Path):
    doc = nbformat.read(path, as_version=4)
    ip = get_ipython()
    executed = 0
    for idx, cell in enumerate(doc.cells):
        if cell.cell_type != "code" or not cell.source.strip():
            continue
        print(f"\n--- v10a.26 code cell {idx+1}/{len(doc.cells)} ---", flush=True)
        result = ip.run_cell(cell.source, store_history=False)
        executed += 1
        if getattr(result, "error_before_exec", None) is not None:
            raise result.error_before_exec
        if getattr(result, "error_in_exec", None) is not None:
            raise result.error_in_exec
    print(f"Executed {executed} v10a.26 code cells.")

missing = missing_base_symbols()
if missing:
    print("Base symbols missing:", len(missing))
    execute_notebook_in_current_kernel(V26_PATH)
    missing = missing_base_symbols()

if missing:
    raise RuntimeError("v10a.26 did not create required symbols: " + ", ".join(missing))

print("v10a.26 namespace firewall: PASS")
print("L =", L, "N =", N)



## Blind order-5 firewall

This first invocation is deliberately short:

- order \(5\)
- Haar occurrence cap \(7\)
- one-face physical regression only
- no support census
- no production shapes

Do not start production if any gate fails.


In [ ]:

def configure_v28(order: int, mode: str, run_census: bool, production: bool):
    order = int(order)
    if order not in (4, 5, 6, 7):
        raise ValueError(order)
    cap = 7 if order <= 5 else 9

    os.environ["V28_ORDER"] = str(order)
    os.environ["V28_MODE"] = mode
    os.environ["V28_HAAR_CAP"] = str(cap)
    os.environ["V28_RUN_CENSUS"] = "1" if run_census else "0"
    os.environ["V28_GPU"] = "1"
    os.environ["V28_GPU_SW_MIN_DIM"] = str(GPU_SW_MIN_DIM)
    os.environ["V28_HERMITICITY_AUDIT_PAIRS"] = str(HERM_AUDIT_PAIRS)
    os.environ["V28_DUPLICATE_CHECKS"] = str(DUPLICATE_CHECKS)
    os.environ["V28_HEARTBEAT"] = str(HEARTBEAT_SECONDS)
    os.environ["V28_RESUME"] = "1"
    os.environ["V28_ALLOW_REFERENCE_REBUILD"] = "0"

    order_dir = WORKDIR / f"m{order}"
    order_dir.mkdir(parents=True, exist_ok=True)
    os.environ["V28_CHECKPOINT"] = str(order_dir / "shapes.pkl")
    os.environ["V28_CENSUS_CHECKPOINT"] = str(order_dir / "census.pkl")

    if production:
        os.environ["V28_PRODUCTION_CONFIRM"] = f"YES_ORDER_{order}"
        if order == 5:
            os.environ["V28_MAX_NEW_SHAPES"] = str(M5_MAX_NEW_SHAPES)
            os.environ["V28_TIME_BUDGET_MINUTES"] = str(M5_TIME_BUDGET_MINUTES)
        else:
            # Higher-order default: one atomic shape per invocation until you tune it.
            os.environ["V28_MAX_NEW_SHAPES"] = "1"
            os.environ["V28_TIME_BUDGET_MINUTES"] = "30"
    else:
        os.environ["V28_PRODUCTION_CONFIRM"] = ""
        os.environ["V28_MAX_NEW_SHAPES"] = "1"
        os.environ["V28_TIME_BUDGET_MINUTES"] = "30"

def exec_v28():
    code = V28_PATH.read_text(encoding="utf-8")
    exec(compile(code, str(V28_PATH), "exec"), globals())

configure_v28(order=5, mode="firewall", run_census=False, production=False)
exec_v28()

if not all(ok for _, ok, _ in V28_GATES):
    raise RuntimeError("ORDER-5 FIREWALL FAILED. Do not start production.")

print("\nORDER-5 FIREWALL: PASS")



## Blind \(m_5\) production

This is the expensive cell.

Settings here remove the old `dt-before-assignment` failure path by using the hardened v10a.28 production loop. The run is atomic per rooted shape and checkpoints after every shape.

With `M5_MAX_NEW_SHAPES=0` and `M5_TIME_BUDGET_MINUTES=0`, it attempts all remaining shapes in one invocation. If Colab disconnects, rerun this cell; it resumes from `shapes.pkl`.


In [ ]:

configure_v28(order=5, mode="production", run_census=True, production=True)
t_m5 = time.time()
exec_v28()
elapsed_m5 = time.time() - t_m5

print(f"\nOrder-5 invocation elapsed: {elapsed_m5/3600:.3f} h")

# V28_RESULT is either a complete result or an intentional budget-stop payload.
if V28_RESULT is None:
    raise RuntimeError("v10a.28 did not return V28_RESULT")

print("V28_RESULT complete:", V28_RESULT.get("complete", True))
if not V28_RESULT.get("complete", True):
    print("Checkpoint saved. Rerun this production cell to continue.")


In [ ]:

# Freeze a compact blind m5 provenance record when production is complete.
def jsonable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    return x

M5_DIR = WORKDIR / "m5"
M5_SUMMARY = M5_DIR / "blind_m5_summary.json"

if V28_RESULT.get("complete", True):
    coeff = np.asarray(V28_RESULT["coefficients"], dtype=float)
    payload = {
        "schema": "hodge-v10a29-blind-m5-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": 5,
        "blind": True,
        "external_requested_order_target_loaded": False,
        "coefficient_vector_m0_to_m5": coeff.tolist(),
        "m5": float(coeff[5]),
        "v10a26_sha256": V26_SHA256,
        "v10a28_sha256": V28_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "concrete_clusters": int(V28_RESULT.get("concrete_clusters", 0)),
        "shape_classes": int(V28_RESULT.get("shapes", 0)),
        "haar_cap": int(V28_RESULT.get("haar_cap", 0)),
        "krylov_depth": int(V28_RESULT.get("krylov_depth", 0)),
        "gates": [
            {"name": n, "passed": bool(ok), "detail": d}
            for n, ok, d in V28_GATES
        ],
        "gpu": name,
        "elapsed_seconds_last_invocation": float(elapsed_m5),
        "checkpoint": os.environ["V28_CHECKPOINT"],
        "census_checkpoint": os.environ["V28_CENSUS_CHECKPOINT"],
    }
    M5_SUMMARY.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print("FROZEN BLIND m5:", repr(float(coeff[5])))
    print("Summary:", M5_SUMMARY)
    print("Summary SHA-256:", sha256_file(M5_SUMMARY))
else:
    print("m5 is not complete yet; no blind coefficient was frozen.")



## External holdout boundary

At this point, if `blind_m5_summary.json` exists:

1. copy it off the runtime;
2. record its SHA-256;
3. compare \(m_5\) to the external published holdout **outside this notebook**;
4. if and only if you accept the result, create the validation flag below.

The published target is intentionally absent here.


In [ ]:

# Run this only after external validation of the frozen blind m5 result.
MARK_M5_EXTERNALLY_VALIDATED = False

flag5 = WORKDIR / "m5" / "M5_EXTERNALLY_VALIDATED.flag"
if MARK_M5_EXTERNALLY_VALIDATED:
    if not M5_SUMMARY.exists():
        raise RuntimeError("Cannot validate: blind_m5_summary.json is missing.")
    flag5.write_text(
        "User marked frozen blind m5 externally validated at "
        + datetime.now(timezone.utc).isoformat() + "\n",
        encoding="utf-8",
    )
    print("Created:", flag5)
else:
    print("m6 remains locked. Set MARK_M5_EXTERNALLY_VALIDATED=True only after external comparison.")



## \(m_6\) and \(m_7\) — deliberately locked

The engine supports orders 6 and 7 with Haar cap 9 and Krylov depth 3.

This notebook does not automatically proceed. That prevents a failed \(m_5\) result from contaminating the next stage.

For \(m_6\), create the `M5_EXTERNALLY_VALIDATED.flag` above.

For \(m_7\), first freeze and externally validate \(m_6\), then create `M6_EXTERNALLY_VALIDATED.flag`.

Higher-order runs default to **one new atomic shape per invocation**. After the first few shape timings are known, you can raise the per-invocation limit without changing the physics.


In [ ]:

def run_higher_order(order: int):
    order = int(order)
    if order not in (6, 7):
        raise ValueError("This helper is only for m6 or m7.")

    if not (WORKDIR / "m5" / "M5_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m6/m7 locked: m5 has not been marked externally validated.")
    if order == 7 and not (WORKDIR / "m6" / "M6_EXTERNALLY_VALIDATED.flag").exists():
        raise RuntimeError("m7 locked: m6 has not been marked externally validated.")

    # Short firewall at the requested order.
    configure_v28(order=order, mode="firewall", run_census=False, production=False)
    exec_v28()
    if not all(ok for _, ok, _ in V28_GATES):
        raise RuntimeError(f"ORDER-{order} FIREWALL FAILED.")

    # Production, resumable. Defaults to one new shape/invocation.
    configure_v28(order=order, mode="production", run_census=True, production=True)
    t0 = time.time()
    exec_v28()
    dt = time.time() - t0
    print(f"Order-{order} invocation elapsed: {dt/3600:.3f} h")

    if not V28_RESULT.get("complete", True):
        print("Incomplete by intentional budget/shape limit. Rerun run_higher_order(order).")
        return V28_RESULT

    coeff = np.asarray(V28_RESULT["coefficients"], dtype=float)
    order_dir = WORKDIR / f"m{order}"
    summary = order_dir / f"blind_m{order}_summary.json"

    # Self-generated lower-order consistency locks.
    lower_checks = {}
    m5_data = json.loads((WORKDIR / "m5" / "blind_m5_summary.json").read_text())
    lower_checks["m5_vs_frozen_m5"] = {
        "current": float(coeff[5]),
        "frozen": float(m5_data["m5"]),
        "abs_error": abs(float(coeff[5]) - float(m5_data["m5"])),
    }
    if order == 7:
        m6_data = json.loads((WORKDIR / "m6" / "blind_m6_summary.json").read_text())
        lower_checks["m6_vs_frozen_m6"] = {
            "current": float(coeff[6]),
            "frozen": float(m6_data["m6"]),
            "abs_error": abs(float(coeff[6]) - float(m6_data["m6"])),
        }

    payload = {
        "schema": f"hodge-v10a29-blind-m{order}-freeze-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "order": order,
        "blind": True,
        "external_requested_order_target_loaded": False,
        f"m{order}": float(coeff[order]),
        "coefficient_vector": coeff.tolist(),
        "lower_order_self_consistency": lower_checks,
        "v10a26_sha256": V26_SHA256,
        "v10a28_sha256": V28_SHA256,
        "v28_schema": V28_RESULT.get("schema"),
        "v28_signature": V28_RESULT.get("signature"),
        "gates": [{"name": n, "passed": bool(ok), "detail": d} for n, ok, d in V28_GATES],
        "gpu": name,
    }
    summary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"FROZEN BLIND m{order}:", repr(float(coeff[order])))
    print("Summary:", summary)
    print("Summary SHA-256:", sha256_file(summary))
    return V28_RESULT

# Examples — leave commented until the validation flags exist:
# run_higher_order(6)
# run_higher_order(7)


In [ ]:

# Package compact provenance/results. Checkpoints are intentionally excluded because they can be very large.
bundle = WORKDIR / "blind_results_compact.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(WORKDIR.rglob("*")):
        if not p.is_file():
            continue
        if p.suffix == ".pkl":
            continue
        z.write(p, p.relative_to(WORKDIR))
    # Include exact source files used.
    z.write(V26_PATH, Path("sources") / V26_PATH.name)
    z.write(V28_PATH, Path("sources") / V28_PATH.name)

print("Compact bundle:", bundle)
print("SHA-256:", sha256_file(bundle))



## What to send back after the A100 run

For \(m_5\), send:

- the final firewall summary;
- `blind_m5_summary.json`;
- the last 100–200 lines around `ROOTED INCIDENCE TRANSFORM`;
- any exception traceback;
- if the run stops intentionally, the `completed_shapes / total_shapes` count and the last completed shape dimensions.

Do **not** paste the external \(m_5\) target into the runtime before the blind summary is frozen.
